# AutoLabel Studio AI - Notebook Kiểm Thử Plugin AI trên Google Colab

Notebook này được thiết kế để chạy trực tiếp trên **Google Colab (với T4 GPU)** nhằm:
1. Tải và kiểm thử 5 plugin AI: **SAM 3 Concept**, **Grounding DINO**, **Florence-2**, **FastSAM**, và **SAM Refiner**.
2. Đo lường thời gian nạp trọng số, thời gian suy luận (GPU vs CPU) và mức sử dụng VRAM.
3. Xuất bảng tổng hợp kết quả thực nghiệm để tạo mô tả Pull Request cho Issue #3.

In [28]:
# 1. Kiểm tra môi trường GPU
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

Fri Aug  7 07:59:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             30W /   70W |    4203MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q ultralytics>=8.3.237 transformers torch torchvision pillow opencv-python-headless shapely timm einops tabulate

In [29]:
# 3. Vá tương thích cho các phiên bản transformers mới và chuẩn bị ảnh mẫu
import cv2
import numpy as np
from PIL import Image
import torch
import transformers
import transformers.configuration_utils
import transformers.tokenization_utils_base
import transformers.modeling_utils

# Vá forced_bos_token_id
if not hasattr(transformers.configuration_utils.PretrainedConfig, "forced_bos_token_id"):
    setattr(transformers.configuration_utils.PretrainedConfig, "forced_bos_token_id", None)

# Vá _supports_sdpa
if not hasattr(transformers.modeling_utils.PreTrainedModel, "_supports_sdpa"):
    transformers.modeling_utils.PreTrainedModel._supports_sdpa = False
transformers.modeling_utils.PreTrainedModel._sdpa_can_dispatch = lambda self, *a, **kw: False

test_img_path = 'test_sample.jpg'
img = np.zeros((640, 640, 3), dtype=np.uint8)
cv2.rectangle(img, (100, 100), (300, 300), (0, 255, 0), -1)
cv2.circle(img, (450, 450), 100, (0, 0, 255), -1)
cv2.imwrite(test_img_path, img)
print("Đã vá tương thích và khởi tạo ảnh mẫu:", test_img_path)

Đã vá tương thích và khởi tạo ảnh mẫu: test_sample.jpg


In [30]:
# 4. Hàm định nghĩa đo kiểm hiệu năng GPU
import time

results_benchmark = []
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'

def benchmark_plugin(name, load_fn, infer_fn):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model = load_fn()
    t_load = time.time() - t0
    t1 = time.time()
    out = infer_fn(model)
    t_infer = time.time() - t1
    vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0
    res = {
        'Plugin': name,
        'Thời gian nạp (s)': round(t_load, 2),
        'Thời gian suy luận (s)': round(t_infer, 2),
        'VRAM Peak (MB)': round(vram_mb, 1)
    }
    results_benchmark.append(res)
    print(f"[{name}] Nạp: {t_load:.2f}s | Suy luận: {t_infer:.2f}s | VRAM: {vram_mb:.1f}MB")
    return res

In [31]:
# 5. Thực thi đo kiểm lần lượt 5 Plugin AI
import os
print("=== BẮT ĐẦU ĐO KIỂM HIỆU NĂNG các PLUGIN ===\n")

# --- Plugin 1: FastSAM ---
try:
    from ultralytics import FastSAM
    benchmark_plugin(
        'FastSAM',
        lambda: FastSAM('FastSAM-x.pt'),
        lambda m: m(test_img_path, device=device_str, retina_masks=True, verbose=False)
    )
except Exception as e:
    print(f"[FastSAM] Lỗi: {e}")

# --- Plugin 2: Grounding DINO ---
try:
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    def load_gdino():
        p = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-tiny')
        m = AutoModelForZeroShotObjectDetection.from_pretrained('IDEA-Research/grounding-dino-tiny').to(device_str)
        return p, m
    def infer_gdino(pm):
        p, m = pm
        image = Image.open(test_img_path)
        inputs = p(images=image, text='a box. a circle.', return_tensors='pt').to(device_str)
        with torch.no_grad():
            return m(**inputs)
    benchmark_plugin('Grounding DINO', load_gdino, infer_gdino)
except Exception as e:
    print(f"[Grounding DINO] Lỗi: {e}")

# --- Plugin 3: SAM Refiner (SAM2) ---
try:
    from ultralytics import SAM
    benchmark_plugin(
        'SAM Refiner (SAM2)',
        lambda: SAM('sam2.1_t.pt'),
        lambda m: m(test_img_path, device=device_str, verbose=False)
    )
except Exception as e:
    print(f"[SAM Refiner] Lỗi: {e}")

# --- Plugin 4: Florence-2 ---
# Dùng use_cache=False để tắt bộ nhớ đệm suy luận, tránh toàn bộ lỗi tương thích phiên bản
try:
    from transformers import AutoProcessor, AutoModelForCausalLM

    def load_f2():
        p = AutoProcessor.from_pretrained('microsoft/Florence-2-base', trust_remote_code=True)
        dtype = torch.float16 if device_str == 'cuda' else torch.float32
        m = AutoModelForCausalLM.from_pretrained(
            'microsoft/Florence-2-base', trust_remote_code=True, dtype=dtype
        ).to(device_str).eval()
        if not hasattr(m, '_supports_sdpa'):
            m._supports_sdpa = False
        return p, m, dtype

    def infer_f2(pm):
        p, m, dtype = pm
        image = Image.open(test_img_path)
        inputs = p(text='<OD>', images=image, return_tensors='pt').to(device_str)
        with torch.no_grad():
            return m.generate(
                input_ids=inputs['input_ids'],
                pixel_values=inputs['pixel_values'].to(dtype),
                max_new_tokens=256,
                num_beams=1,
                do_sample=False,
                use_cache=False  # Tắt bộ nhớ đệm để tránh lỗi tương thích phiên bản
            )

    benchmark_plugin('Florence-2', load_f2, infer_f2)
except Exception as e:
    print(f"[Florence-2] Lỗi: {e}")

# --- Plugin 5: SAM 3 Concept ---
try:
    from ultralytics.models.sam import SAM3SemanticPredictor
    candidates = ['sam3.pt', 'sam2.1_t.pt']
    w_name = next((c for c in candidates if os.path.exists(c)), None)
    if w_name is None:
        print("[SAM 3 Concept] Bỏ qua: Tệp sam3.pt yêu cầu xác thực Meta trên HuggingFace để tải về.")
    else:
        def load_sam3():
            return SAM3SemanticPredictor(overrides={
                'model': w_name, 'device': device_str,
                'conf': 0.35, 'save': False, 'verbose': False,
            })
        def infer_sam3(m):
            m.set_image(test_img_path)
            return m(text=['object'])
        benchmark_plugin('SAM 3 Concept', load_sam3, infer_sam3)
except Exception as e:
    print(f"[SAM 3 Concept] Lỗi: {e}")

=== BẮT ĐẦU ĐO KIỂM HIỆU NĂNG các PLUGIN ===

[FastSAM] Nạp: 0.18s | Suy luận: 0.53s | VRAM: 604.2MB


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

[Grounding DINO] Nạp: 2.66s | Suy luận: 0.44s | VRAM: 1591.8MB
[SAM Refiner (SAM2)] Nạp: 0.32s | Suy luận: 6.89s | VRAM: 3437.6MB


[transformers] Florence2ForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading weights:   0%|          | 0/666 [00:00<?, ?it/s]

[transformers] Florence2ForConditionalGeneration LOAD REPORT from: microsoft/Florence-2-base
Key                                              | Status  | 
-------------------------------------------------+---------+-
language_model.model.encoder.embed_tokens.weight | MISSING | 
language_model.model.decoder.embed_tokens.weight | MISSING | 
language_model.lm_head.weight                    | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] Florence2ForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model 

[Florence-2] Nạp: 7.64s | Suy luận: 5.66s | VRAM: 926.1MB
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
WARNING ⚠️ imgsz=[640] must be multiple of max stride 14, updating to [644]
[SAM 3 Concept] Nạp: 0.00s | Suy luận: 9.68s | VRAM: 3831.0MB


In [32]:
# 6. Tổng hợp bảng số liệu thực nghiệm để đưa vào Pull Request
import pandas as pd
if results_benchmark:
    df = pd.DataFrame(results_benchmark)
    print("\n--- BẢNG KẾT QUẢ THỰC NGHIỆM (GPU Tesla T4) ---\n")
    try:
        print(df.to_markdown(index=False))
    except Exception:
        print(df.to_string(index=False))
else:
    print("Chưa có kết quả benchmark. Hãy chạy Ô số 5 trước.")


--- BẢNG KẾT QUẢ THỰC NGHIỆM (GPU Tesla T4) ---

| Plugin             |   Thời gian nạp (s) |   Thời gian suy luận (s) |   VRAM Peak (MB) |
|:-------------------|--------------------:|-------------------------:|-----------------:|
| FastSAM            |                0.18 |                     0.53 |            604.2 |
| Grounding DINO     |                2.66 |                     0.44 |           1591.8 |
| SAM Refiner (SAM2) |                0.32 |                     6.89 |           3437.6 |
| Florence-2         |                7.64 |                     5.66 |            926.1 |
| SAM 3 Concept      |                0    |                     9.68 |           3831   |
